In [ ]:
#Basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile

#DL/CV imports
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset

from PIL import Image

#Select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Running on', device)

Running on cuda


In [ ]:
#Extract data
zip_name = 'drawn_words.zip'
zip_ref = zipfile.ZipFile(zip_name, 'r')
zip_ref.extractall()
zip_ref.close()

#Load data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
train_df.head()

,image_path,label
0,output_dataset/train/000000.png,violin
1,output_dataset/train/000001.png,moon
2,output_dataset/train/000002.png,violin
3,output_dataset/train/000003.png,book
4,output_dataset/train/000004.png,tree


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
train_df['label'] = encoder.fit_transform(train_df['label'])

num_classes = len(train_df['label'].unique())

class ImageDataset(Dataset):
    def __init__(self, df, transformer=None, has_labels=False):
        self.df = df
        self.transformer = transformer
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image_path = row['image_path']

        image = Image.open(image_path).convert('L')  #Gray scale

        if self.transformer:
            image = self.transformer(image)

        if self.has_labels:
            label = int(row['label'])
            return image, label

        return image

#Define transformers
mean = [0.5,0.5,0.5]
std = [0.5,0.5,0.5]

train_transformer = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((300,300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transformer = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((300,300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

#Creating loaders
train_dataset = ImageDataset(train_df, transformer = train_transformer, has_labels=True)
test_dataset = ImageDataset(test_df, transformer = test_transformer, has_labels=False)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
#Loading model
model = models.efficientnet_b2(weights='DEFAULT')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model.to(device)
print()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

epochs=15
best_val_acc=0.0

for epoch in range(epochs):
    model.train()
    training_loss=0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        training_loss += loss.item()

    training_loss /=len(train_loader)
    print(f'Epoch{epoch+1}: Loss={training_loss:.4f}')


1: Loss=0.2062
2: Loss=0.1659
3: Loss=0.1164
4: Loss=0.1004
5: Loss=0.0533
6: Loss=0.0348
7: Loss=0.0249
8: Loss=0.0143
9: Loss=0.0261
10: Loss=0.0156
11: Loss=0.0110
12: Loss=0.0120
13: Loss=0.0124
14: Loss=0.0092
15: Loss=0.0071


In [ ]:
#Predicting

model.eval()
preds = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, batch_preds = torch.max(outputs, 1)

        preds.extend(batch_preds.cpu().numpy())

In [ ]:
#Submitting
preds = encoder.inverse_transform(preds)

output_df = pd.DataFrame({
    'image_path':test_df['image_path'],
    'label':preds
})

output_df.to_csv('submission.csv', index=False)

Final score: 100; Accuracy: 0.98